OpenRouteService (ORS) provides a free API for routing, distance matrix, geocoding, route optimization etc. using OpenStreetMap data

In [ ]:
ORS_API_KEY = 'YOUR_API_KEY'

In [2]:
import requests

san_francisco = (37.7749, -122.4194)
new_york = (40.661, -73.944)

parameters = {
    'api_key': ORS_API_KEY,
    'start' : '{},{}'.format(san_francisco[1], san_francisco[0]),
    'end' : '{},{}'.format(new_york[1], new_york[0])
}

# we get the driving distance using Directions API
response = requests.get(
    'https://api.openrouteservice.org/v2/directions/driving-car', params=parameters)

if response.status_code == 200:
    print('Request successful.')
    data = response.json()
else:
    print('Request failed.')
    print('Reason', response.text) 

Request successful.


In [3]:
#data

In [4]:
summary = data['features'][0]['properties']['summary']
print(summary)

{'distance': 4692944.5, 'duration': 167337.4}


In [5]:
distance = summary['distance']
print(distance/1000) #distance in km

4692.9445


In [6]:
from geopy import distance as geopy_distance
#approssimando la Terra a una sfera (Haversine's formula)
straight_line_distance = geopy_distance.great_circle(san_francisco, new_york)
#considerando la forma ellissoidale della Terra con l'ellissoide WGS-84 (Vincenty's formula)
ellipsoid_distance = geopy_distance.geodesic(san_francisco, new_york, ellipsoid='WGS-84')

print(straight_line_distance, ellipsoid_distance)

4135.3804590061345 km 4145.446977549561 km


Come si vede, ovviamente, la distanza geodesica è minore della distanza stradale

Calcoliamo ora il tempo di guida in minuti ed ore

In [7]:
duration = summary['duration'] #è in secondi
duration_minutes = duration / 60
print(duration_minutes)
duration_hours = duration_minutes / 60
print(duration_hours)

2788.9566666666665
46.482611111111105


EXERCISE 2: for loop and use of time library to avoid too many requests in a short time

Note: the limit of the free plan is 40 direction requests/minute

In [8]:
import time

def get_driving_distance(source_coordinates, dest_coordinates):
    parameters = {
    'api_key': ORS_API_KEY,
    'start' : '{},{}'.format(source_coordinates[1], source_coordinates[0]),
    'end' : '{},{}'.format(dest_coordinates[1], dest_coordinates[0])
    }

    response = requests.get(
        'https://api.openrouteservice.org/v2/directions/driving-car', params=parameters)

    if response.status_code == 200:
        data = response.json()
        summary = data['features'][0]['properties']['summary']
        distance = summary['distance']
        return distance/1000
    else:
        print('Request failed.')
        print('Reason', response.text) 
        return -9999

san_francisco = (37.7749, -122.4194)

destination_cities = {
    'Los Angeles': (34.0522, -118.2437),
    'Boston': (42.3601, -71.0589),
    'Atlanta': (33.7490, -84.3880)
}

for city, coordinates in destination_cities.items():
    distance = get_driving_distance(san_francisco, coordinates)
    print(f'Distance from San Francisco to {city}: {distance:.2f} km')
    time.sleep(2) #pausa di 2 secondo tra le richieste

Distance from San Francisco to Los Angeles: 615.52 km
Distance from San Francisco to Boston: 4986.67 km
Distance from San Francisco to Atlanta: 3976.32 km
